> **Research record.** Cleaned copy of `800-research-users-locations/3000-pu-modelling.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Positive-unlabelled (Elkan-Noto) model that scores Google POIs as third places, with validation.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

### After we collected the sample with groundtruth we try to identify third places in unlabelled data

In [ ]:
import sys
from pathlib import Path

prepare_data_path = Path().absolute().parent
if str(prepare_data_path) not in sys.path:
    sys.path.insert(0, str(prepare_data_path))

In [ ]:
from geofunctions import utils

In [ ]:
df = utils.read_query_to_dataframe(
    "select * from `phd-habidatum.phd_pois.google_pois_features`"
)

In [ ]:
TARGET_VAR="flag_keyword_review_rw"

In [ ]:
# they highly correlate with rank_cosine_similarity_work_morning and dont add much information
# we remove them and dont mention them in the paper
drop_cols = [
    "flag_groundtruth",  # it is old manual labels
    "avg_rw_visits_share_percentile_0_1", # duplicate with avg_rw_visits_share_weekday,
    "avg_cosine_similarity_work_morning",
    "avg_cosine_similarity_work_late_morning",
    "avg_cosine_similarity_work_afternoon",
]

In [ ]:
df.drop(columns=drop_cols, inplace=True)

In [ ]:
df = df[
    (df["flag_location_suits_rw"] == 1)
    & (df["flag_transport_hub_risk"] == 0)
    & (df["flag_category_suits_rw"] == 1)
]

In [ ]:
df.drop(columns=['flag_location_suits_rw','flag_transport_hub_risk','flag_category_suits_rw'],inplace=True)

In [ ]:
df = df[df['userratingcount']>0]

In [ ]:
identifier_cols=['id', 'name', 'primarytype', 'geometry', 'geohash', 'yishuv_stat_2022',
       'census_new_neighborhood_type']

In [ ]:
data_source_map = {
    # Word2Vec results
    "avg_cosine_similarity_work_morning": "Word2Vec results",
    "avg_cosine_similarity_work_late_morning": "Word2Vec results",
    "avg_cosine_similarity_work_afternoon": "Word2Vec results",
    "rank_cosine_similarity_work_morning": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning": "Word2Vec results",
    "rank_cosine_similarity_work_afternoon": "Word2Vec results",
    "rank_cosine_similarity_work_morning_same_neighborhood_type_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_morning_satellite_city_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_morning_cbd_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_morning_residential_central_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_morning_residential_remote_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning_same_neighborhood_type_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning_satellite_city_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning_cbd_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning_residential_central_in_primarytype": "Word2Vec results",
    "rank_cosine_similarity_work_late_morning_residential_remote_in_primarytype": "Word2Vec results",

    # Google Places API
    "pricerange": "Google Places API",
    "livemusic": "Google Places API",
    "servesdessert": "Google Places API",
    "restroom": "Google Places API",
    "servescoffee": "Google Places API",
    "userratingcount": "Google Places API",
    "rating": "Google Places API",

    # GPS-signals analysis
    "min_norm_share_10_15": "GPS-signals analysis",
    "max_n_signals": "GPS-signals analysis",
    "hourly_signal_std": "GPS-signals analysis",
    "hourly_signal_std_percentile_by_keyword": "GPS-signals analysis",
    "avg_rw_visits_share_after2022": "GPS-signals analysis",
    "avg_rw_visits_share_percentile_0_1": "GPS-signals analysis",
    "remote_workers_visits_share_weekday": "GPS-signals analysis",
    "share_dwells_home_lt_1km_rw": "GPS-signals analysis",
    "avg_dist_home_rw": "GPS-signals analysis",
    "visit_frequency_rw": "GPS-signals analysis",
    "avg_duration_hours_weekday_rw": "GPS-signals analysis",
    "n_visits_weekday_rw": "GPS-signals analysis",
    "avg_duration_hours_weekday": "GPS-signals analysis",
    "share_friday": "GPS-signals analysis",
    "share_weekday": "GPS-signals analysis",
    "share_dwells_home_lt_1km": "GPS-signals analysis",
    "visit_frequency": "GPS-signals analysis",
    "avg_dist_home_work_ratio": "GPS-signals analysis",
    "avg_dist_home": "GPS-signals analysis",
    "n_visits_weekday": "GPS-signals analysis",
    "flag_rw_visits_share_after_2022": "GPS-signals analysis",
    "flag_office_substitute_daily_dynamic": "GPS-signals analysis",
    "share_visits_after_home": "GPS-signals analysis",
    "share_visits_after_work": "GPS-signals analysis",

    # Google reviews / derived POI labels
    "flag_atmosphere_suits_rw": "Google Places API",
    "flag_keyword_review_rw": "Google Places API",
    "offices_proximity": "Google Places API",
}

In [ ]:
df.rename(columns = {"office_probability_risk":"offices_proximity"},inplace=True)

In [ ]:
df_stat_nulls = df.drop(columns=identifier_cols).isnull().mean().to_frame(name='null_share').sort_values('null_share', ascending=False)
df_stat_nulls.reset_index(inplace=True)
df_stat_nulls.rename(columns={"index": "feature_name"}, inplace=True)

In [ ]:
df_stat_nulls['data_source'] = df_stat_nulls['feature_name'].map(data_source_map)

In [ ]:
#potential third places features
ok_features = df_stat_nulls[(df_stat_nulls['null_share'] < 0.53)&(df_stat_nulls['data_source']!='Word2Vec results')&(df_stat_nulls['feature_name'].apply(lambda x: (('_rw' in x)&('atmosphere'not in x))==False))].feature_name.tolist()

## Replace nulls

In [ ]:
import pandas as pd

# Missing-value replacement (see null_share table above).
# Strategies: "rank_mean_and_zero" = two rank features: mean-imputed + zero-imputed; "zero"; "mean".

IMPUTATION_STRATEGIES = {
    # Word2Vec results — no evidence is represented in two ways for sensitivity.
    "rank_cosine_similarity_work_morning": "rank_mean_and_zero",
    "rank_cosine_similarity_work_late_morning": "rank_mean_and_zero",
    "rank_cosine_similarity_work_afternoon": "rank_mean_and_zero",
    # Google Places API
    "pricerange": "mean",
    "livemusic": "zero",
    "servesdessert": "zero",
    "restroom": "zero",
    "servescoffee": "zero",
    # GPS-signals analysis
    "avg_rw_visits_share_after2022": "zero",
    "visit_frequency_rw": "zero",
    "remote_workers_visits_share_weekday": "zero",
    "avg_dist_home_rw": "mean",
    "share_dwells_home_lt_1km_rw": "mean",
    "avg_duration_hours_weekday_rw": "mean",
    "n_visits_weekday_rw": "zero",
    "n_visits_weekday": "zero",
    "avg_duration_hours_weekday": "mean",
    "share_friday": "mean",
    "share_weekday": "mean",
    "share_dwells_home_lt_1km": "mean",
    "visit_frequency": "zero",
    "avg_dist_home_work_ratio": "mean",
    "avg_dist_home": "mean",
    "min_norm_share_10_15": "mean",
    "max_n_signals": "mean",
    "hourly_signal_std_percentile_by_keyword": "mean",
    "hourly_signal_std": "mean",
    "share_visits_after_home": "mean",
    "share_visits_after_work": "mean",
}

RANK_COLS = [c for c, s in IMPUTATION_STRATEGIES.items() if s == "rank_mean_and_zero"]
RANK_ZERO_COLS = [f"{c}_zero_imputed" for c in RANK_COLS]


def _coerce_feature(series, col):
    """Convert feature to numeric; pricerange may be a Google Places price dict."""
    if col == "pricerange":

        def avg_price_range(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return pd.NA
            if isinstance(x, dict):
                start_obj = x.get("startPrice") or {}
                end_obj = x.get("endPrice") or {}
                start_units = start_obj.get("units") if isinstance(start_obj, dict) else None
                end_units = end_obj.get("units") if isinstance(end_obj, dict) else None
                if start_units is None and end_units is None:
                    return pd.NA
                if start_units is None:
                    return float(end_units)
                if end_units is None:
                    return float(start_units)
                return (float(start_units) + float(end_units)) / 2
            return x

        return pd.to_numeric(series.apply(avg_price_range), errors="coerce").astype("float64")
    return pd.to_numeric(series, errors="coerce").astype("float64")


for col, strategy in IMPUTATION_STRATEGIES.items():
    if col not in df.columns:
        raise KeyError(f"Missing column for imputation: {col}")

    values = _coerce_feature(df[col], col)

    if strategy == "rank_mean_and_zero":
        zero_col = f"{col}_zero_imputed"
        stale_missing_col = f"{col}_missing"

        # Remove old missing-flag features from the previous imputation strategy.
        if stale_missing_col in df.columns:
            df.drop(columns=stale_missing_col, inplace=True)
            data_source_map.pop(stale_missing_col, None)

        df[col] = values.fillna(values.mean())

        # If the cell is rerun after mean imputation, keep the existing zero-imputed feature.
        if zero_col in df.columns and values.isna().sum() == 0:
            df[zero_col] = pd.to_numeric(df[zero_col], errors="coerce").fillna(0)
        else:
            df[zero_col] = values.fillna(0)
        data_source_map[zero_col] = "Word2Vec results"
    elif strategy == "zero":
        df[col] = values.fillna(0)
    elif strategy == "mean":
        df[col] = values.fillna(values.mean())
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

# Quick check: nulls remaining in imputed columns (should all be 0)
imputed_cols = list(IMPUTATION_STRATEGIES) + RANK_ZERO_COLS

## Dataset

In [ ]:
feature_cols = [
    c
    for c in ok_features
    if c not in identifier_cols and c!=TARGET_VAR   and c!='remote_workers_visits_share_weekday' #and c!='min_norm_share_10_15' 
    
]
X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.astype("float32")

y = df.loc[X.index, TARGET_VAR].astype(int).values

## The Elkan & Noto Method (2008) with elastic net

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


class ElkanNotoPU:
    """
    Positive-Unlabeled (PU) Learning using the Elkan & Noto (2008) method.
    """

    def __init__(self, estimator=None):
        # Default to Logistic Regression as used in the original paper
        self.estimator = estimator if estimator is not None else LogisticRegression()
        self.c_ = None  # The estimated probability that a positive is labeled

    def fit(self, X, s):
        """
        Fit the PU model.
        X: Feature matrix
        s: Labels (1 for known positive, 0 for unlabeled)
        """
        # Step 1: Train a standard classifier to predict P(s=1 | x)
        self.estimator.fit(X, s)

        # Step 2: Estimate 'c' (the probability that a true positive is labeled)
        # We use estimator 'e1' from the paper: the average predicted probability
        # of being labeled, evaluated only on the known positive examples.
        probs = self.estimator.predict_proba(X)[:, 1]
        known_positive_probs = probs[s == 1]

        self.c_ = np.mean(known_positive_probs)

        return self

    def predict_proba(self, X):
        """
        Predict the true probability P(y=1 | x)
        """
        if self.c_ is None:
            raise ValueError("The model is not fitted yet.")

        # Get the probability of being labeled P(s=1 | x)
        probs_s = self.estimator.predict_proba(X)[:, 1]

        # Adjust probabilities: P(y=1 | x) = P(s=1 | x) / c
        probs_y = probs_s / self.c_

        # Probabilities can occasionally exceed 1.0 due to estimation variance, so clip them
        return np.clip(probs_y, 0.0, 1.0)

    def predict(self, X, threshold=0.5):
        """
        Predict the binary class (0 or 1) based on a threshold.
        """
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

### PU validation without true negatives

These checks are designed for the scientific-paper setting where we have trusted positives and an unlabeled population, but no reliable true-negative sample. Therefore we do **not** report ordinary accuracy, precision, or recall against unlabeled observations as if they were negatives. Instead, we test whether the PU model can recover hidden positives, whether known positives are enriched at the top of the ranking, whether results are stable across random splits, and whether conclusions are robust to elastic-net settings.

In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp


def make_elkan_noto_model(C=0.1, l1_ratio=0.5, random_state=42):
    """Create a fresh Elkan-Noto PU model with elastic-net logistic regression."""
    estimator = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "logreg",
                LogisticRegression(
                    penalty="elasticnet",
                    l1_ratio=l1_ratio,
                    C=C,
                    solver="saga",
                    max_iter=2000,
                    random_state=random_state,
                ),
            ),
        ]
    )
    return ElkanNotoPU(estimator=estimator)


def positive_holdout_validation(
    X,
    y,
    holdout_frac=0.3,
    n_repeats=100,
    top_k_values=(25, 50, 100, 250),
    C=0.1,
    l1_ratio=0.5,
    random_seed=42,
    feature_names=None,
    return_coefficients=False,
):
    """
    Validate PU ranking without true negatives.

    Scientific interpretation:
    - We hide a random subset of trusted positives and train on the remaining positives.
    - Hidden positives are treated as unlabeled during training.
    - A good discovery model should rank hidden positives near the top.
    - This estimates recovery of known-positive structure, not full precision/recall against true negatives.
    """
    X_eval = X.reset_index(drop=True) if hasattr(X, "reset_index") else X
    feature_names = list(feature_names) if feature_names is not None else (
        list(X_eval.columns) if hasattr(X_eval, "columns") else None
    )
    y_eval = np.asarray(y)
    positive_idx = np.where(y_eval == 1)[0]
    n_holdout = max(1, int(round(holdout_frac * len(positive_idx))))
    rng = np.random.default_rng(random_seed)
    records = []
    coefficient_records = []

    for repeat in range(n_repeats):
        hidden_idx = rng.choice(positive_idx, size=n_holdout, replace=False)
        s_observed = y_eval.copy()
        s_observed[hidden_idx] = 0

        model = make_elkan_noto_model(C=C, l1_ratio=l1_ratio, random_state=random_seed + repeat)
        model.fit(X_eval, s_observed)
        scores = model.predict_proba(X_eval)

        ranking = np.argsort(-scores)
        rank_position = np.empty_like(ranking)
        rank_position[ranking] = np.arange(1, len(scores) + 1)
        hidden_ranks = rank_position[hidden_idx]

        record = {
            "repeat": repeat,
            "c_hat": model.c_,
            "hidden_positive_median_rank": np.median(hidden_ranks),
            "hidden_positive_median_percentile": np.median(1 - (hidden_ranks - 1) / len(scores)),
            "hidden_positive_mean_score": scores[hidden_idx].mean(),
            "unlabeled_mean_score": scores[y_eval == 0].mean(),
        }

        for k in top_k_values:
            top_k_idx = set(ranking[: min(k, len(ranking))])
            record[f"recall_at_{k}"] = np.mean([idx in top_k_idx for idx in hidden_idx])

        records.append(record)

        if return_coefficients:
            logreg = model.estimator.named_steps["logreg"]
            coefs = logreg.coef_.ravel()
            names = feature_names or [f"feature_{i}" for i in range(len(coefs))]
            coefficient_records.extend(
                {
                    "repeat": repeat,
                    "C": C,
                    "l1_ratio": l1_ratio,
                    "feature": feature,
                    "coef_scaled": coef,
                    "abs_coef": abs(coef),
                }
                for feature, coef in zip(names, coefs)
            )

    results = pd.DataFrame(records)
    if return_coefficients:
        coefficients = pd.DataFrame(coefficient_records)
        return results, coefficients

    return results


def hidden_positive_enrichment(
    X,
    y,
    holdout_frac=0.3,
    n_repeats=30,
    C_values=(0.03, 0.1, 0.3, 1.0),
    l1_ratio_values=(0.25, 0.5, 0.75),
    top_share_step=0.01,
    random_seed=42,
):
    """
    Ranking enrichment for hidden positives (positive holdout).

    Mirrors the full-sample enrichment curve, but:
    - 30% of trusted positives are hidden and treated as unlabeled during training.
    - Recovery is measured only among hidden positives in the ranked list.
    """
    X_eval = X.reset_index(drop=True) if hasattr(X, "reset_index") else X
    y_eval = np.asarray(y)
    positive_idx = np.where(y_eval == 1)[0]
    n_holdout = max(1, int(round(holdout_frac * len(positive_idx))))
    rng = np.random.default_rng(random_seed)
    rows = []

    for repeat in range(n_repeats):
        hidden_idx = rng.choice(positive_idx, size=n_holdout, replace=False)
        hidden_mask = np.zeros(len(y_eval), dtype=bool)
        hidden_mask[hidden_idx] = True

        s_observed = y_eval.copy()
        s_observed[hidden_idx] = 0
        n_hidden = len(hidden_idx)
        baseline_rate = n_hidden / len(y_eval)

        for C_value in C_values:
            for l1_ratio_value in l1_ratio_values:
                model = make_elkan_noto_model(
                    C=C_value,
                    l1_ratio=l1_ratio_value,
                    random_state=random_seed + repeat,
                )
                model.fit(X_eval, s_observed)
                scores = model.predict_proba(X_eval)
                ranking = np.argsort(-scores)

                for share in np.arange(0, 1.01, top_share_step):
                    k = max(1, int(round(share * len(y_eval))))
                    top_idx = ranking[:k]
                    hidden_in_top = int(hidden_mask[top_idx].sum())
                    rows.append(
                        {
                            "repeat": repeat,
                            "holdout_frac": holdout_frac,
                            "C": C_value,
                            "l1_ratio": l1_ratio_value,
                            "top_share": share,
                            "top_k": k,
                            "hidden_positives_in_top_k": hidden_in_top,
                            "n_hidden_positives": n_hidden,
                            "share_of_hidden_positives_recovered": hidden_in_top / n_hidden,
                            "hidden_positive_rate_in_top_k": hidden_in_top / k,
                            "baseline_hidden_positive_rate": baseline_rate,
                            "enrichment_over_baseline": (hidden_in_top / k) / baseline_rate,
                        }
                    )

    return pd.DataFrame(rows)



# Main paper-style validation: repeated positive holdout recovery.
# This does not require true negatives and should be reported as a ranking validation.
holdout_results, holdout_coefficients = positive_holdout_validation(
    X,
    y,
    holdout_frac=0.3,
    n_repeats=100,
    top_k_values=(25, 50, 100, 250),
    C=0.1,
    l1_ratio=0.5,
    random_seed=42,
    feature_names=feature_cols,
    return_coefficients=True,
)

holdout_summary = holdout_results.agg(["mean", "std"])

In [ ]:
# Average coefficients across positive-holdout validation repeats.
# Coefficients are on the standardized feature scale because the model includes StandardScaler.
holdout_coef_summary = (
    holdout_coefficients.groupby(["feature", "C", "l1_ratio"], as_index=False)
    .agg(
        mean_coef_scaled=("coef_scaled", "mean"),
        std_coef_scaled=("coef_scaled", "std"),
        mean_abs_coef=("abs_coef", "mean"),
        nonzero_share=("coef_scaled", lambda s: (s != 0).mean()),
    )
    .sort_values("mean_abs_coef", ascending=False)
)

In [ ]:
ax = holdout_coef_summary.loc[holdout_coef_summary['mean_abs_coef'] > 0.092, ['feature', 'mean_coef_scaled']].set_index('feature').sort_values('mean_coef_scaled', ascending=True).plot(kind='barh', legend=False)
ax.set_title('Mean Standardized Coefficient (Selected Features, Holdout PU Model)')
ax.set_xlabel('Mean Coefficient (Standardized)')
ax.set_ylabel('Feature')

In [ ]:
from matplotlib import pyplot as plt
holdout_results['hidden_positive_median_percentile'].hist(bins=20)
plt.title("Distribution of Hidden Positive Median Percentile (Holdout Repeats)")
plt.xlabel("Hidden Positive Median Percentile")
plt.ylabel("Frequency")

In [ ]:
# Ranking enrichment test.
# Question: are trusted positives disproportionately concentrated in the top-scored places?
# This is appropriate without true negatives because it only compares ranking concentration.

y_array = np.asarray(y)
n_positives = y_array.sum()
baseline_rate = n_positives / len(y_array)

C_VALUES = [0.03, 0.1, 0.3, 1.0]
L1_RATIO_VALUES = [0.25, 0.5, 0.75]

enrichment_rows = []
for C_value in C_VALUES:
    for l1_ratio_value in L1_RATIO_VALUES:
        model = make_elkan_noto_model(
            C=C_value, l1_ratio=l1_ratio_value, random_state=42
        )
        model.fit(X, y)
        scores = model.predict_proba(X)

        # Keep default setting for downstream score-separation cell
        if C_value == 0.1 and l1_ratio_value == 0.5:
            full_scores = scores

        ranking = np.argsort(-scores)
        for share in np.arange(0, 1.01, 0.01):
            k = max(1, int(round(share * len(y_array))))
            top_idx = ranking[:k]
            positives_in_top = y_array[top_idx].sum()
            enrichment_rows.append(
                {
                    "C": C_value,
                    "l1_ratio": l1_ratio_value,
                    "top_share": share,
                    "top_k": k,
                    "known_positives_in_top_k": int(positives_in_top),
                    "share_of_all_known_positives_recovered": positives_in_top / n_positives,
                    "known_positive_rate_in_top_k": positives_in_top / k,
                    "baseline_known_positive_rate": baseline_rate,
                    "enrichment_over_baseline": (positives_in_top / k) / baseline_rate,
                }
            )

enrichment_df = pd.DataFrame(enrichment_rows)

In [ ]:
# Hidden-positive ranking enrichment (30% positive holdout).
# Same curve as above, but positives are hidden during training and evaluated only on the holdout set.
HOLDOUT_FRAC = 0.3
N_REPEATS = 30

hidden_enrichment_df = hidden_positive_enrichment(
    X,
    y,
    holdout_frac=HOLDOUT_FRAC,
    n_repeats=N_REPEATS,
    C_values=C_VALUES,
    l1_ratio_values=L1_RATIO_VALUES,
    random_seed=42,
)

hidden_enrichment_mean_df = (
    hidden_enrichment_df.groupby(["C", "l1_ratio", "top_share"], as_index=False)
    .agg(
        top_k=("top_k", "first"),
        hidden_positives_in_top_k=("hidden_positives_in_top_k", "mean"),
        n_hidden_positives=("n_hidden_positives", "first"),
        share_of_hidden_positives_recovered=(
            "share_of_hidden_positives_recovered",
            "mean",
        ),
        share_of_hidden_positives_recovered_std=(
            "share_of_hidden_positives_recovered",
            "std",
        ),
        hidden_positive_rate_in_top_k=("hidden_positive_rate_in_top_k", "mean"),
        baseline_hidden_positive_rate=("baseline_hidden_positive_rate", "first"),
        enrichment_over_baseline=("enrichment_over_baseline", "mean"),
    )
)

In [ ]:
import matplotlib.pyplot as plt

hidden_plot_df = (
    hidden_enrichment_mean_df.sort_values(["C", "l1_ratio", "top_share"])
    .assign(
        settings=lambda d: "C="
        + d["C"].astype(str)
        + ", l1="
        + d["l1_ratio"].astype(str)
    )
    .pivot(
        index="top_share",
        columns="settings",
        values="share_of_hidden_positives_recovered",
    )
)

fig, ax = plt.subplots(figsize=(10, 6))
hidden_plot_df.plot(ax=ax, legend=True)
ax.set_title(
    f"Recall of Hidden Positives vs. Top Share (holdout={HOLDOUT_FRAC}, mean over {N_REPEATS} repeats)"
)
ax.set_xlabel("Top Share")
ax.set_ylabel("Share of Hidden Positives Recovered")
ax.legend(title="Model settings", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()


In [ ]:
# Hidden-positive ranking enrichment (30% positive holdout).
# Same curve as above, but positives are hidden during training and evaluated only on the holdout set.
HOLDOUT_FRAC = 0.3
N_REPEATS = 30

hidden_enrichment_df = hidden_positive_enrichment(
    X,
    y,
    holdout_frac=HOLDOUT_FRAC,
    n_repeats=N_REPEATS,
    C_values=C_VALUES,
    l1_ratio_values=L1_RATIO_VALUES,
    random_seed=42,
)

hidden_enrichment_mean_df = (
    hidden_enrichment_df.groupby(["C", "l1_ratio", "top_share"], as_index=False)
    .agg(
        top_k=("top_k", "first"),
        hidden_positives_in_top_k=("hidden_positives_in_top_k", "mean"),
        n_hidden_positives=("n_hidden_positives", "first"),
        share_of_hidden_positives_recovered=(
            "share_of_hidden_positives_recovered",
            "mean",
        ),
        share_of_hidden_positives_recovered_std=(
            "share_of_hidden_positives_recovered",
            "std",
        ),
        hidden_positive_rate_in_top_k=("hidden_positive_rate_in_top_k", "mean"),
        baseline_hidden_positive_rate=("baseline_hidden_positive_rate", "first"),
        enrichment_over_baseline=("enrichment_over_baseline", "mean"),
    )
)

### Not used in the paper

In [ ]:
# Score separation test.
# Question: do trusted positives receive systematically higher PU scores than unlabeled places?
# We do not call unlabeled places negatives; these tests only check distributional separation.
positive_scores = full_scores[y_array == 1]
unlabeled_scores = full_scores[y_array == 0]

mannwhitney = mannwhitneyu(positive_scores, unlabeled_scores, alternative="greater")
ks_test = ks_2samp(positive_scores, unlabeled_scores, alternative="two-sided")

score_separation = pd.DataFrame(
    [
        {
            "group": "known_positive",
            "n": len(positive_scores),
            "mean_score": positive_scores.mean(),
            "median_score": np.median(positive_scores),
            "p25_score": np.percentile(positive_scores, 25),
            "p75_score": np.percentile(positive_scores, 75),
        },
        {
            "group": "unlabeled",
            "n": len(unlabeled_scores),
            "mean_score": unlabeled_scores.mean(),
            "median_score": np.median(unlabeled_scores),
            "p25_score": np.percentile(unlabeled_scores, 25),
            "p75_score": np.percentile(unlabeled_scores, 75),
        },
    ]
)

print(f"Mann-Whitney U p-value, positives > unlabeled: {mannwhitney.pvalue:.4g}")
print(f"Kolmogorov-Smirnov p-value, different distributions: {ks_test.pvalue:.4g}")

In [ ]:
# Top-K stability test.
# Question: do repeated positive-holdout models select similar top candidate places?
# Stable top candidates are more credible for a discovery/ranking paper.
def top_k_stability(
    X,
    y,
    holdout_frac=0.3,
    n_repeats=50,
    top_k=100,
    C=0.1,
    l1_ratio=0.5,
    random_seed=42,
):
    X_eval = X.reset_index(drop=True) if hasattr(X, "reset_index") else X
    y_eval = np.asarray(y)
    positive_idx = np.where(y_eval == 1)[0]
    n_holdout = max(1, int(round(holdout_frac * len(positive_idx))))
    rng = np.random.default_rng(random_seed)
    top_sets = []

    for repeat in range(n_repeats):
        hidden_idx = rng.choice(positive_idx, size=n_holdout, replace=False)
        s_observed = y_eval.copy()
        s_observed[hidden_idx] = 0

        model = make_elkan_noto_model(C=C, l1_ratio=l1_ratio, random_state=random_seed + repeat)
        model.fit(X_eval, s_observed)
        scores = model.predict_proba(X_eval)
        top_sets.append(set(np.argsort(-scores)[:top_k]))

    jaccards = []
    for i in range(len(top_sets)):
        for j in range(i + 1, len(top_sets)):
            union_size = len(top_sets[i] | top_sets[j])
            jaccards.append(len(top_sets[i] & top_sets[j]) / union_size if union_size else np.nan)

    return pd.DataFrame(
        [
            {
                "top_k": top_k,
                "n_repeats": n_repeats,
                "mean_pairwise_jaccard": np.nanmean(jaccards),
                "std_pairwise_jaccard": np.nanstd(jaccards),
                "min_pairwise_jaccard": np.nanmin(jaccards),
            }
        ]
    )

stability_df = top_k_stability(
    X,
    y,
    holdout_frac=0.3,
    n_repeats=50,
    top_k=100,
    C=0.1,
    l1_ratio=0.5,
    random_seed=42,
)

In [ ]:
# Elastic-net sensitivity analysis.
# Question: are conclusions similar under reasonable regularization choices?
# For a paper, report whether recall@K and c_hat are stable across model settings.
sensitivity_rows = []
for C_value in [0.03, 0.1, 0.3, 1.0]:
    for l1_ratio_value in [0.25, 0.5, 0.75]:
        result = positive_holdout_validation(
            X,
            y,
            holdout_frac=0.3,
            n_repeats=30,
            top_k_values=(50, 100, 250),
            C=C_value,
            l1_ratio=l1_ratio_value,
            random_seed=42,
        )
        sensitivity_rows.append(
            {
                "C": C_value,
                "l1_ratio": l1_ratio_value,
                "mean_c_hat": result["c_hat"].mean(),
                "mean_recall_at_50": result["recall_at_50"].mean(),
                "mean_recall_at_100": result["recall_at_100"].mean(),
                "mean_recall_at_250": result["recall_at_250"].mean(),
                "mean_hidden_positive_median_percentile": result[
                    "hidden_positive_median_percentile"
                ].mean(),
            }
        )

sensitivity_df = pd.DataFrame(sensitivity_rows).sort_values(
    ["mean_recall_at_100", "mean_hidden_positive_median_percentile"], ascending=False
)

## Final specification to produce results

In [ ]:

X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
X = X.astype("float32")

y = df.loc[X.index, TARGET_VAR].astype(int).values

In [ ]:
# Final model for downstream prediction and coefficient interpretation.
# The validation cells above repeatedly hide positives on purpose. After validation,
# retrain on all trusted positives so later cells use the intended production/research model.
pu_model = make_elkan_noto_model(C=0.03, l1_ratio=0.5, random_state=42)
pu_model.fit(X, y)

print(f"Final Elkan-Noto c_hat using all trusted positives: {pu_model.c_:.3f}")

In [ ]:
logreg = pu_model.estimator.named_steps["logreg"]
coef = logreg.coef_.ravel()  # shape (n_features,)
intercept = logreg.intercept_[0]

coef_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "coef_scaled": coef,
        "abs_coef": np.abs(coef),
    }
).sort_values("abs_coef", ascending=False)

In [ ]:
df.loc[X.index, "prediction_third_place"] = pu_model.predict(X)

In [ ]:
df.loc[X.index, "predict_prob"] = pu_model.predict_proba(X)

In [ ]:
df['primarytype_agg'] = df['primarytype'].apply(lambda x: 'specialty_cafe' if x in(['cat_cafe','dessert_restaurant','dog_cafe','tea_house','breakfast_restaurant','brunch_restaurant']) else x)

In [ ]:
import matplotlib.pyplot as plt

pivot_tbl = pd.pivot_table(
    df,
    values='prediction_third_place',
    index='census_new_neighborhood_type',
    columns='primarytype_agg',
    aggfunc='sum',
    fill_value=0
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
# cmap='YlGnBu' gives a nice yellow-to-blue gradient
sns.heatmap(pivot_tbl, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=.5)

plt.title('Heatmap of Primary Types across Neighborhood Types')
plt.xlabel('Primary Type')
plt.ylabel('Neighborhood Type')
plt.tight_layout()
plt.show()

In [ ]:
pivot_tbl_shr = pd.pivot_table(
    df,
    values='prediction_third_place',
    index='census_new_neighborhood_type',
    columns='primarytype_agg',
    aggfunc='mean',
    fill_value=0
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
# Convert proportions to percentages for display
percentages = pivot_tbl_shr * 100

# Use integer or float format (without the % sign, since that's not supported by the heatmap)
sns.heatmap(percentages, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=.5,
            cbar_kws={'label': 'Percentage (%)'})

plt.title('Heatmap of Primary Types across Neighborhood Types (Percentages)')
plt.xlabel('Primary Type')
plt.ylabel('Neighborhood Type')
plt.tight_layout()
plt.show()

In [ ]:
import os
import sys
from pyproj import datadir

# Ensure PROJ uses the database from this conda env
env_proj = os.path.join(sys.prefix, "share", "proj")
if not os.path.isdir(env_proj):
    raise RuntimeError(f"PROJ data dir not found at {env_proj}")

# Set both env vars and pyproj's internal data dir
os.environ["PROJ_DATA"] = env_proj
os.environ["PROJ_LIB"] = env_proj
datadir.set_data_dir(env_proj)

print("Using PROJ data from:", env_proj)
print("pyproj data dir:", datadir.get_data_dir())

In [ ]:
gdf = utils.make_gdf(df,geometry="geometry",crs="4326")

In [ ]:
utils.upload_table_to_bq(
    gdf,
    dataset="phd_models_results",
    table_name="google_3rd_places_detected_pu_20260601",
    if_table_exist="replace"
)